## **Análise de série de vazões (Xingó)**

Aristides Guilherme da Silva

### 1 - Leitura dos dados e gerando hidrogramas

In [1]:
from hidrocomp.series.flow import Flow, Partial
import numpy as np
import plotly as py
import plotly.io as pio
import plotly.graph_objects as go
import pandas as pd

Teste lib local


### **2 - Séries de durações parciais**

#### 2.1 - Separando eventos hidrológicos para calibração e validação da cópula (1931 - 1994)

In [128]:
file = "ONS_daily_flow.csv"
dados = pd.read_csv(file, index_col=0, parse_dates=True)
dados = pd.DataFrame(dados["XINGO (178)"])

#Filtrando período de análise ()
#dados = dados[(dados.index >= "1931-09-01") & (dados.index <= "1994-08-31")]

dados_pre_impacto = dados[(dados.index <= "1994-08-31")]

#Convetendo obj pandas para Series Flow
dados_pre_impacto = Flow(pd.DataFrame(dados_pre_impacto["XINGO (178)"]))

station = "XINGO (178)"

#'stationary' or 'events_by_year'
type_threshold = 'stationary'  

#'flood' or 'drought'
type_event = 'flood'

#'media', 'mediana' or 'autocorrelation' 
type_criterion = 'autocorrelation'

#Peaks or percentil
value_threshold = 0.75

#calculando sdp 
sdp_pre_impact = dados_pre_impacto.partial(type_threshold, type_event, type_criterion, value_threshold, duration=0)

#salvando gráfico da sdp do período pré-impacto, ou seja, antes da implementação da hidrelétrica de Xingó
fig_hidro_sdp_pre_impact, data_hidro_sdp_pre_impact = sdp_pre_impact.plot_hydrogram(title='SDP Pré-Impacto (Xingo)')
#py.offline.plot(fig, filename='gráficos/figura2_histo_sdp_pre_impacto_1931_2018.html')
pio.show(fig_hidro_sdp_pre_impact)

#apenas salvando um dataframe
df_sdp_pre_impacto = sdp_pre_impact.information

#adicionando datas em convertidas para dados circulares (ou direcionais) em graus
df_sdp_pre_impacto['Date_circular'] = sdp_pre_impact.julian_radius()
df_sdp_pre_impacto

,Duration,End,Julian,Peaks,Start,TR,Date_circular
Date,,,,,,,
1931-01-11,23,1931-01-24,11,5753.0,1931-01-01,2.283197,130.191781
1931-05-06,92,1931-05-19,126,7527.0,1931-02-16,5.463855,243.616438
1932-02-09,30,1932-02-20,40,5493.0,1932-01-21,1.946352,158.360656
1932-12-29,19,1933-01-11,364,4912.0,1932-12-23,1.428909,117.049180
1933-02-20,44,1933-02-28,51,6476.0,1933-01-15,2.988468,169.643836
...,...,...,...,...,...,...,...
1992-05-04,3,1992-05-05,125,5003.0,1992-05-02,1.502900,241.967213
1993-01-19,92,1993-02-16,19,6977.0,1992-11-16,3.763485,138.082192
1993-03-04,32,1993-03-22,63,5745.0,1993-02-18,2.247831,181.479452


In [75]:
len(sdp_pre_impact.information)

145

#### 2.2 - Separando eventos hidrológicos para validação da cópula (1995 - 2018)


In [100]:
#===================== LENDO DADOS ===========================
file = "ONS_daily_flow.csv"
data_pos_impacto = pd.read_csv(file, index_col=0, parse_dates=True)
data_pos_impacto = pd.DataFrame(data_pos_impacto ["XINGO (178)"])

#apenas filtrando para o período de após a implementação da hidrelétrica de Xingó
data_pos_impacto = data_pos_impacto [(data_pos_impacto .index >= "1994-09-01") & (data_pos_impacto .index <= "2018-08-31")]

#Convetendo obj pandas para Series Flow
data_pos_impacto = Flow(pd.DataFrame(data_pos_impacto))

#===================== SDPs ===========================
station = "XINGO (178)"

#'stationary' or 'events_by_year'
type_threshold = 'stationary' 

#'flood' or 'drought'
type_event = 'flood'

#'media', 'mediana' or 'autocorrelation' 
type_criterion = 'autocorrelation'
value_threshold = 4060 #tive que baixar 8 m3/s do limiar da série entre 1931 e 1994

#gerando sdp
sdp_pos_impacto = data_pos_impacto.partial(type_threshold, type_event, type_criterion, value_threshold, duration=1)

fig_hidro_sdp_pos_impact, data_hidro_sdp_pos_impact = sdp_pos_impacto.plot_hydrogram(title='SDP Pós-Impacto (Xingo)')
#py.offline.plot(fig_hidro_sdp_pos_impact, filename='gráficos/figura3_histo_sdp_pos_impacto_1931_2018.html')
#pio.show(fig_hidro_sdp_pos_impact)

#apenas salvando um dataframe
df_sdp_pos_impacto = sdp_pos_impacto.information

#adicionando datas em convertidas para dados circulares (ou direcionais) em graus
df_sdp_pos_impacto['Date_circular'] = sdp_pos_impacto.julian_radius()

#Filtrando eventos que o Clebson apresentou tabela na dissertação
df_sdp_pos_impacto.loc[['1996-01-17', '1998-01-03', '2000-01-18', '2000-04-03', '2002-01-18', '2004-01-18', '2004-01-31', '2005-05-02']]


,Duration,End,Julian,Peaks,Start,TR,Date_circular
Date,,,,,,,
1996-01-17,34,1996-01-29,17,5481.0,1995-12-26,1.860853,135.737705
1998-01-03,12,1998-01-08,3,5297.0,1997-12-27,1.633550,122.301370
2000-01-18,18,2000-01-26,18,5527.0,2000-01-08,1.951362,136.721311
2000-04-03,26,2000-04-12,94,5843.0,2000-03-17,2.284738,211.475410
2002-01-18,23,2002-01-27,18,7373.0,2002-01-04,8.798246,137.095890
2004-01-18,1,2004-01-19,18,4326.0,2004-01-18,1.097374,136.721311
2004-01-31,108,2004-05-07,31,12194.0,2004-01-20,71.642857,149.508197
2005-05-02,1,2005-05-03,122,4602.0,2005-05-02,1.195471,239.671233


In [112]:
#salvando picos de vazões dos eventos
magnitude = sdp_pos_impacto.information['Peaks']

#estimando parâmetros via momentos-l
sdp_pos_impacto.dist_gpa.mml()

#calculando 
p = sdp_pos_impacto.dist_gpa.probs(magnitude)

#quantidade média de eventos por ano
mu = len(sdp_pos_impacto.information)/(2018-1994)

#calculando tr univariado
tr_uni = mu / (1 - p)

#salvando resultados
df_sdp_pos_impact_results = df_sdp_pos_impacto[['Date_circular', 'Duration', 'Peaks']].copy()
df_sdp_pos_impact_results['TR_Uni'] = tr_uni

#visualizando
df_sdp_pos_impact_results.loc[['1996-01-17', '1998-01-03', '2000-01-18', '2000-04-03', '2002-01-18', '2004-01-18', '2004-01-31', '2005-05-02']]


,Date_circular,Duration,Peaks,TR_Uni
Date,,,,
1996-01-17,135.737705,34,5481.0,3.579712
1998-01-03,122.301370,12,5297.0,3.174261
2000-01-18,136.721311,18,5527.0,3.689558
2000-04-03,211.475410,26,5843.0,4.549568
2002-01-18,137.095890,23,7373.0,13.191239
2004-01-18,136.721311,1,4326.0,1.713080
2004-01-31,149.508197,108,12194.0,765.120040
2005-05-02,239.671233,1,4602.0,2.035391


### **3 - Modelagem das distribuições marginais**

#### 3.1 - Magnitude (Generalizada de Pareto)

In [4]:
#Distribuição cumulativa
title = 'Xingó'
type_function = 'density'

fig_cumulative_mml_pareto, data_cumulative_mmml_pareto = sdp_validation.plot_distribution(title, type_function, estimador='MML')
pio.show(fig_cumulative_mml_pareto)


#### 3.2 - Duração ( Pearson III) 

In [7]:
#Distribuição densidade-probabilidade
title = 'Xingó'
type_function = 'density'
fig_acum_mml_pearson, data_acum_mmml_pareto = sdp_calibration.plot_distribution(title, type_function, estimador='mml', distribuition='P3', variable='Duration')
py.offline.plot(fig_acum_mml_pearson, filename='gráficos/figura5a_dist_cum_duration_mml.html')
pio.show(fig_acum_mml_pearson)

#Distribuição densidade-probabilidade
title = 'Xingó'
type_function = 'cumulative'
fig_density_mml_pearson, data_density_mmml_pareto = sdp_calibration.plot_distribution(title, type_function, estimador='mml', distribuition='P3', variable='Duration')
py.offline.plot(fig_density_mml_pearson, filename='gráficos/figura5b_dist_density_duration_mml.html')
pio.show(fig_density_mml_pearson)


#### 3.3 - Período de ocorrência (Von-Mises)

In [7]:
#Criando objetivo para manipular os dados de período de ocorrência de cheias 
period_calibration = Circular(df_sdp_calibration)

#Visualizando gráfico de barras circular
period_calibration.plot_bar_circular(month_num_start_year_hydrologic=9, unit='degrees')

#Gráfico de dispersão
period_calibration.plot_scatter_circular(month_num_start_year_hydrologic=9, unit='degrees')

In [33]:
from scipy.stats import vonmises_line

circular_dates = period_calibration.circular_date(9, 'rad')

vonmises_line.fit(circular_dates)

(0.553914260906888, 2.976129335223642, 0.4992439586508519)

In [204]:
from lmoments3 import distr


In [34]:
circular_dates

1932-02-09    2.763915
1932-12-29    2.048491
1933-02-20    2.960843
1934-02-07    2.737059
1935-02-24    3.029700
                ...   
1992-05-04    4.223125
1993-01-19    2.409989
1993-03-04    3.167414
1994-02-11    2.805916
1994-04-04    3.701054
Name: date_peaks, Length: 138, dtype: float64

#### 3.4 Parâmetros das distribuições marginais

##### Pré impacto

In [147]:
from lmoments3.distr import gpa, pe3
from scipy.stats import vonmises_line

#magnitude
peaks_GP_params_pre_impact = [gpa.lmom_fit(df_sdp_pre_impacto['Peaks'])['c'],
                   gpa.lmom_fit(df_sdp_pre_impacto['Peaks'])['loc'],
                   gpa.lmom_fit(df_sdp_pre_impacto['Peaks'])['scale']]

#duração
duration_P3_params_pre_impact = [pe3.lmom_fit(df_sdp_pre_impacto['Duration'])['skew'],
                      pe3.lmom_fit(df_sdp_pre_impacto['Duration'])['loc'],
                      pe3.lmom_fit(df_sdp_pre_impacto['Duration'])['scale']]

#periodo de ocorrênica
period_vonmises_params_pre_impact = [vonmises_line.fit(df_sdp_pre_impacto['Date_circular'])[0],
                          vonmises_line.fit(df_sdp_pre_impacto['Date_circular'])[1],
                          vonmises_line.fit(df_sdp_pre_impacto['Date_circular'])[2]]

#Visualizando
print(f'Peaks - GP  {peaks_GP_params_pre_impact}')
print('Duration - P3 ', duration_P3_params_pre_impact)
print('Period - VM ', period_vonmises_params_pre_impact)

Peaks - GP  [np.float64(0.18466794196875164), np.float64(4132.866147073268), np.float64(1832.3458273088893)]
Duration - P3  [2.1829250770911086, np.float64(40.09655172413793), np.float64(43.46863174578785)]
Period - VM  [np.float64(0.5486961896223206), np.float64(170.63014014406622), np.float64(28.569403456776797)]


### **4 - Modelagem função cópula Gaussiana (abordagem frequencista)**

In [7]:
correlation_matrix = np.array([
    [1.0, 0.941, 0.232],
    [0.941, 1.0, 0.155],
    [0.232, 0.155, 1.0]
])

#### 4.1 Cópulas para período pré impacto das  (1931 - 1994)

In [148]:
from copulae import GaussianCopula
from scipy.stats import genpareto, pearson3, vonmises_line, norm, multivariate_normal, vonmises

ModuleNotFoundError: No module named 'copulae'

##### 4.1.1 Cópula trivariada (Pico, Duração e Período)

In [9]:
#Adicionando ao dataframe os dados circulares
df_sdp_calibration['circular_period'] = period_calibration.circular_date(9, 'degrees')

#Selecionando amostras
X1_calibration = df_sdp_calibration.peaks.to_list()
X2_calibration = df_sdp_calibration.Duration.to_list()
X3_calibration = df_sdp_calibration.circular_period.to_list()

#Transformando distribuição marginais para o espaço uniforme (por meio da CDF)
u1_calibration = genpareto.cdf(X1_calibration , peaks_GP_params[0], peaks_GP_params[1], peaks_GP_params[2])
u2_calibration = pearson3.cdf(X2_calibration , duration_P3_params[0], duration_P3_params[1], duration_P3_params[2])
u3_calibration = vonmises_line.cdf(X3_calibration, period_vonmises_params[0], period_vonmises_params[1], period_vonmises_params[2])

#Transformando distribuição marginais para o espaço uniforme (por meio da CDF)
#u1_calibration = genpareto.cdf(X1_calibration, 0.19, 4112.91,  1835.72)
#u2_calibration = pearson3.cdf(X2_calibration, 2.25, 40.5, 44.57)
#u3_calibration = vonmises_line.cdf(X3_calibration, 0.56, 170.67, 28.58)

#Guardando valores para calibração
data_calibration = pd.DataFrame()
data_calibration['peaks'], data_calibration['duration'], data_calibration['period'] = u1_calibration, u2_calibration, u3_calibration

#Extraindo parâmetros cópula
copule_gaussian_tri = GaussianCopula(dim=3)
copule_gaussian_tri.fit(data_calibration, method='irho')

#Definindo relação de dependência entre as variáveis
#copule_gaussian_tri.params = np.array([0.787, 0.164, 0.117])

#copule_gaussian_tri.params = correlation_matrix
copule_gaussian_tri.summary()

            peaks    duration      period
count  138.000000  138.000000  138.000000
mean     0.500000    0.500000    0.500000
std      0.287626    0.287579    0.287620
min      0.021583    0.021583    0.007194
25%      0.255396    0.258993    0.254496
50%      0.500000    0.500000    0.500000
75%      0.746403    0.746403    0.741007
max      0.992806    0.992806    0.992806


1.000000,0.905373,0.215746
0.905373,1.000000,0.069576
0.215746,0.069576,1.000000


In [296]:
stats.spearmanr(data_calibration['duration'], data_calibration['peaks'])


SignificanceResult(statistic=0.8972061661710694, pvalue=4.0389555560269445e-50)

##### 4.1.2 Cópula bivariada (Duração e Período)

In [10]:
#Extraindo parâmetros cópula bivariada
copule_gaussian_bi = GaussianCopula(dim=2)
#copule_gaussian_bi.fit(data_calibration[['duration', 'period']])

copule_gaussian_bi.params = np.array([0.102176])


copule_gaussian_bi.summary()

1.000000,0.102176
0.102176,1.000000


#### 4.2 Separando dados validação das cópulas (1994 - 2018)

In [11]:
#Criando objetivo para manipular os dados de período de ocorrência de cheias 
period_validation = Circular(df_sdp_validation)

#Calculando datas circuales
df_sdp_validation['circular_period'] = period_validation.circular_date(9, 'degrees')

#Selecionando amostras
X1_validation = df_sdp_validation.peaks.to_list()
X2_validation = df_sdp_validation.Duration.to_list()
X3_validation = df_sdp_validation.circular_period.to_list()

#Transformando distribuição marginais para o espaço uniforme (por meio da CDF)
u1_validation = genpareto.cdf(X1_validation , peaks_GP_params[0], peaks_GP_params[1], peaks_GP_params[2])
u2_validation = pearson3.cdf(X2_validation , duration_P3_params[0], duration_P3_params[1], duration_P3_params[2])
u3_validation = vonmises_line.cdf(X3_validation, period_vonmises_params[0], period_vonmises_params[1],period_vonmises_params[2])


#u1_validation = genpareto.cdf(X1_validation, 0.19, 4112.91,  1835.72)
#u2_validation = pearson3.cdf(X2_validation, 2.25, 40.5, 44.57)
#u3_validation = vonmises_line.cdf(X3_validation, 0.56, 170.67, 28.58)


#Guardando valores para calibração
data_validation = pd.DataFrame()
data_validation['peaks'], data_validation['duration'], data_validation['period'] = u1_validation, u2_validation, u3_validation

#### 4.4 Tempo de retorno multivariado (Picos como variável de referência)

In [12]:
#Calculando CDF multivariadas
cdf_copule_tri = copule_gaussian_tri.cdf(data_validation)
cdf_copule_bi = copule_gaussian_bi.cdf(data_validation[['duration', 'period']])

#Calculando mu
## quantidade de eventos observados e intervalo (quantidade) de anos em que esses eventos ocorreram
n_events = len(df_sdp_validation)
n_years = df_sdp_validation.index.year.max() - df_sdp_validation.index.year.min()

mu = n_events/n_years

#Calculando tempo de retorno multivariado
tr_copule_peaks = mu / (1 - (cdf_copule_tri/cdf_copule_bi))

#Calculando tempo de retorno univariado (pico)
tr_univariate_peaks = mu / (1 - u1_validation)

#Salvando e visualizando resultados
df_sdp_validation['tr_mult'] = tr_copule_peaks
df_sdp_validation['tr_uni'] = tr_univariate_peaks 
df_sdp_validation

,Duration,Start,End,peaks,circular_period,tr_mult,tr_uni
1995-03-02,14,1995-02-22,1995-03-08,5328.00,179.506849,34.084059,3.299276
1996-01-17,35,1995-12-25,1996-01-29,5481.00,135.737705,13.499473,3.547588
1996-12-09,9,1996-12-04,1996-12-13,4729.00,97.643836,16.087697,2.459431
1997-01-24,47,1997-01-06,1997-02-22,7502.00,143.013699,66.343566,8.531995
1997-04-01,44,1997-03-12,1997-04-25,5904.00,209.095890,12.365907,4.314586
1998-01-03,14,1997-12-25,1998-01-08,5297.00,122.301370,27.134371,3.250737
1998-03-09,12,1998-03-03,1998-03-15,4680.00,186.410959,8.561207,2.399313
1999-01-20,2,1999-01-19,1999-01-21,4044.00,139.068493,1.809524,1.809524
1999-03-22,23,1999-03-14,1999-04-06,5985.00,199.232877,47.664785,4.475836
1999-12-22,2,1999-12-21,1999-12-23,4295.00,110.163934,13.697048,1.967429


In [14]:
df_sdp_validation.to_excel('resultados_tr.xlsx')

In [13]:
#Calculando CDF multivariadas
cdf_copule_tri = copule_gaussian_tri.cdf(data_validation)
cdf_copule_bi = copule_gaussian_bi.cdf(data_validation[['duration', 'period']])

#Calculando mu
## quantidade de eventos observados e intervalo (quantidade) de anos em que esses eventos ocorreram
n_events = len(df_sdp_validation)
n_years = df_sdp_validation.index.year.max() - df_sdp_validation.index.year.min()

#mu = n_events/n_years
mu = 1.65

#Calculando tempo de retorno multivariado
tr_copule_peaks = mu / (1 - (cdf_copule_tri/cdf_copule_bi))

#Calculando tempo de retorno univariado (pico)
tr_univariate_peaks = mu / (1 - u1_validation)

#Salvando e visualizando resultados
df_sdp_validation['tr_mult'] = tr_copule_peaks
df_sdp_validation['tr_uni'] = tr_univariate_peaks 
df_sdp_validation

,Duration,Start,End,peaks,circular_period,tr_mult,tr_uni
1995-03-02,14,1995-02-22,1995-03-08,5328.00,179.506849,31.063019,3.008419
1996-01-17,35,1995-12-25,1996-01-29,5481.00,135.737705,12.309059,3.234840
1996-12-09,9,1996-12-04,1996-12-13,4729.00,97.643836,14.690977,2.242612
1997-01-24,47,1997-01-06,1997-02-22,7502.00,143.013699,60.518580,7.779832
1997-04-01,44,1997-03-12,1997-04-25,5904.00,209.095890,11.274705,3.934221
1998-01-03,14,1997-12-25,1998-01-08,5297.00,122.301370,24.744467,2.964159
1998-03-09,12,1998-03-03,1998-03-15,4680.00,186.410959,7.806235,2.187795
1999-01-20,2,1999-01-19,1999-01-21,4044.00,139.068493,1.650000,1.650000
1999-03-22,23,1999-03-14,1999-04-06,5985.00,199.232877,43.449648,4.081256
1999-12-22,2,1999-12-21,1999-12-23,4295.00,110.163934,12.478729,1.793985
